# Notate snapshots and events as MEI

`snapshot_to_mei` writes a table of peaks as a score; `events_to_mei` writes
each onset of an events table as a chord, with the onset times under the
staff. This notebook compares layouts and pitch grids on `conga3` and
`cello_single`, saves the long `speech1` sequence as MEI and SVG, then
notates the onsets of `conga_multidft2_dropped`. Read `visual_tsv` first for
table kinds.

In [ ]:
from pathlib import Path
import audiospylt as asp

## Chord or sequence

`layout="chord"` (the default) stacks all peaks in one chord on a grand staff;
`layout="sequence"` writes one note per measure in row order.

In [ ]:
conga = asp.read_table("../data/tables/conga3.tsv")
print(f"conga3: {len(conga)} peaks")
for layout in ("chord", "sequence"):
    print(f"layout={layout!r}")
    asp.nb.show_mei(asp.snapshot_to_mei(conga, layout=layout))

## Pitch grid and cents labels

`resolution` sets the grid each peak is rounded to: `"semitone"`, `"quarter"`
(the default) or `"eighth"`. Eighth uses the quarter-tone grid and always adds
cents labels: the distance from each written note to the measured frequency.
`asp.pitch_table` lists the same rounding as a table; its `residual` column is
the label.

In [ ]:
for res in ("semitone", "quarter", "eighth"):
    altered = (asp.pitch_table(conga.freq, resolution=res).accid != "").sum()
    print(f"resolution={res!r}: {altered} of {len(conga)} notes altered")
    asp.nb.show_mei(asp.snapshot_to_mei(conga, resolution=res))
pt = asp.pitch_table(conga.freq)
print("cents_label:", [asp.cents_label(r) for r in pt.residual.head(5)])
pt.round(1)

On the quarter-tone grid most notes carry a quarter-tone sharp (`1qs`, `3qs`
in the `accid` column). The eighth score writes the same notes as the quarter
score and adds the labels, here from −20.4 to 24.5 cents. `cents_label` formats
the first five residuals as `+21`, `-13`, `+6`, `-8`, `+5`.

## Spelling and labels

`spelling="flat"` writes the same pitches with flats, and `labels=True` adds
cents labels at any resolution. Hover a notehead to highlight the note and its
label. `help(asp.snapshot_to_mei)` lists `clef`, `naturals` and `title`.

In [ ]:
cello = asp.read_table("../data/tables/cello_single.tsv")
print(f"cello_single: {len(cello)} peaks")
for spelling in ("sharp", "flat"):
    print(f"spelling={spelling!r}")
    asp.nb.show_mei(asp.snapshot_to_mei(cello, layout="sequence", spelling=spelling, labels=True))

## A long sequence: speech1

`speech1` has 41 peaks, many of them a semitone or two apart, which crowds a
chord; write it as a sequence and save it as MEI and as SVG under `mei/`.

In [ ]:
speech = asp.read_table("../data/tables/speech1.tsv")
mei = asp.snapshot_to_mei(speech, layout="sequence", labels=True)
Path("mei").mkdir(exist_ok=True)
Path("mei/speech1_sequence.mei").write_text(mei)
Path("mei/speech1_sequence.svg").write_text(asp.render_svg(mei))
pages = asp.render_svg_pages(mei)
print(f"{len(speech)} peaks; wrote mei/speech1_sequence.mei and .svg; {len(pages)} SVG page(s)")
asp.nb.show_mei(mei)
asp.nb.show_svg(pages[-1])

Verovio fits this sequence on one page; a longer score would give one SVG
string per page. `show_svg` displays the page text when saving or embedding is
needed.

## Events with a time ruler

`events_to_mei` writes one chord per onset and stores the onset times in the
MEI; `render_svg` and `nb.show_mei` draw them as a time ruler under each
system (`render_svg(..., timeline=False)` leaves it out). `time_labels=True`
also prints each onset time above its chord. `colors=True` paints each onset's
chord and its ruler connector with the same colour. Each chord fills
`round(span / measure_seconds)` measures, at least one, where span is the time
to the next onset. The first row of `conga_multidft2_dropped` is a 0 Hz
placeholder, which `events_to_mei` refuses, so keep the rows above 0 Hz.
Before you run the cell: how many chords will the score have?

In [ ]:
raw = asp.read_table("../data/tables/conga_multidft2_dropped.tsv")
events = raw[raw.freq_start > 0]
print(f"{len(raw)} rows, {len(events)} above 0 Hz, onsets {sorted(events.time_start.unique().tolist())} s")
for measure_seconds in (0.5, 0.1):
    mei = asp.events_to_mei(events, measure_seconds=measure_seconds, time_labels=True, colors=True)
    print(f"measure_seconds={measure_seconds}: {mei.count('<measure ')} measures")
Path("mei").mkdir(exist_ok=True)
Path("mei/conga_events.mei").write_text(mei)
asp.nb.show_mei(mei)

4 onsets, 4 chords. With `measure_seconds=0.5` every chord fits in one
measure; with 0.1 (shown) the chord at 0.2 s fills 2 measures and the one at
0.4 s fills 3, so the spacing of the score follows time.